In [3]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb

In [5]:
# Load Documents

documents_path = "../dataset/documents"

documents = []
file_names = []

for file in os.listdir(documents_path):
    if file.endswith(".txt"):
        with open(os.path.join(documents_path, file), "r", encoding="utf-8") as f:
            documents.append(f.read())
            file_names.append(file)

print("Documents Loaded Successfully!")
print("Total Documents:", len(documents))

Documents Loaded Successfully!
Total Documents: 10


In [6]:
# Generate Text Embeddings

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print("Embeddings Generated Successfully!")
print("Embedding Shape:", embeddings.shape)

np.save("../dataset/embeddings.npy", embeddings)

print("Embeddings Saved Successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6513.68it/s]


Embeddings Generated Successfully!
Embedding Shape: (10, 384)
Embeddings Saved Successfully!


In [7]:
# Store Embeddings in ChromaDB

client = chromadb.PersistentClient(path="../chroma_db")

collection = client.get_or_create_collection(
    name="education_documents"
)

ids = [f"doc_{i+1}" for i in range(len(documents))]

collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()
)

print("Documents Stored Successfully!")
print("Total Stored Documents:", collection.count())

Documents Stored Successfully!
Total Stored Documents: 20


In [8]:
# Load Queries

queries_df = pd.read_csv("../dataset/queries.csv")

print("Queries Loaded Successfully!")
print(queries_df)

print("Total Queries:", len(queries_df))

Queries Loaded Successfully!
                                               Query
0                  How does education help students?
1          What are the benefits of online learning?
2  How is artificial intelligence used in education?
3               Why is classroom learning important?
4             How does technology improve education?
5        What is the importance of higher education?
6      How do teachers evaluate student performance?
7                       What is inclusive education?
8       What are the benefits of distance education?
9       How can students improve their study habits?
Total Queries: 10


In [9]:
# Semantic Search

query = "What are the benefits of online learning?"

query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

for i, document in enumerate(results["documents"][0]):
    print(f"Result {i+1}:")
    print(document)
    print()

Result 1:
Online learning allows students to study through
the internet. It provides flexibility, saves time,
and gives access to digital learning resources.
Students can attend classes from anywhere using
computers and mobile devices.

Result 2:
Online learning allows students to study through
the internet. It provides flexibility, saves time,
and gives access to digital learning resources.
Students can attend classes from anywhere using
computers and mobile devices.

Result 3:
Educational technology improves the learning
experience through digital tools and applications.
Smart boards, educational videos, and online
platforms help students understand complex topics
and learn more effectively.

Result 4:
Educational technology improves the learning
experience through digital tools and applications.
Smart boards, educational videos, and online
platforms help students understand complex topics
and learn more effectively.

Result 5:
Distance education allows students to learn
without atte

In [10]:
print("Total Documents in ChromaDB:", collection.count())
print("Retrieved Results:", len(results["documents"][0]))

Total Documents in ChromaDB: 20
Retrieved Results: 5


In [11]:
client.delete_collection("education_documents")

collection = client.create_collection(
    name="education_documents"
)

ids = [f"doc_{i+1}" for i in range(len(documents))]

collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()
)

print("Total Stored Documents:", collection.count())

Total Stored Documents: 10


In [12]:
# Keyword Search

query = "online learning"

keyword_results = []

for i, document in enumerate(documents):
    if query.lower() in document.lower():
        keyword_results.append(document)

print("Keyword Search Results:")

for i, document in enumerate(keyword_results):
    print(f"Result {i+1}:")
    print(document)
    print()

Keyword Search Results:
Result 1:
Online learning allows students to study through
the internet. It provides flexibility, saves time,
and gives access to digital learning resources.
Students can attend classes from anywhere using
computers and mobile devices.



In [13]:
# Compare Semantic Search and Keyword Search

query = "online learning"

query_embedding = model.encode([query])

semantic_results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

print("SEMANTIC SEARCH RESULTS")
for i, document in enumerate(semantic_results["documents"][0]):
    print(f"{i+1}. {document[:100]}")

print("\nKEYWORD SEARCH RESULTS")
for i, document in enumerate(keyword_results):
    print(f"{i+1}. {document[:100]}")

SEMANTIC SEARCH RESULTS
1. Online learning allows students to study through
the internet. It provides flexibility, saves time,

2. Distance education allows students to learn
without attending a physical classroom. It uses
online c
3. Educational technology improves the learning
experience through digital tools and applications.
Smar
4. Traditional classroom learning allows students
to interact directly with teachers and classmates.
It
5. Artificial Intelligence is a technology that
enables machines to perform tasks that normally
require

KEYWORD SEARCH RESULTS
1. Online learning allows students to study through
the internet. It provides flexibility, saves time,



In [14]:
# Retrieve Results for All Queries

retrieved_data = []

for query in queries_df.iloc[:, 0]:
    query_embedding = model.encode([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=5
    )

    for i, document in enumerate(results["documents"][0]):
        retrieved_data.append({
            "Query": query,
            "Rank": i + 1,
            "Retrieved_Document": document
        })

retrieved_df = pd.DataFrame(retrieved_data)

retrieved_df.to_csv(
    "../dataset/retrieved_results.csv",
    index=False
)

print("Retrieval Results Saved Successfully!")
print("Total Retrieved Results:", len(retrieved_df))

Retrieval Results Saved Successfully!
Total Retrieved Results: 50


In [15]:
# Final Results

print("VECTOR EMBEDDINGS PROJECT SUMMARY")
print("--------------------------------")
print("Total Documents:", len(documents))
print("Embedding Shape:", embeddings.shape)
print("ChromaDB Documents:", collection.count())
print("Total Queries:", len(queries_df))
print("Total Retrieved Results:", len(retrieved_df))

display(retrieved_df.head(10))

VECTOR EMBEDDINGS PROJECT SUMMARY
--------------------------------
Total Documents: 10
Embedding Shape: (10, 384)
ChromaDB Documents: 10
Total Queries: 10
Total Retrieved Results: 50


,Query,Rank,Retrieved_Document
0,How does education help students?,1,"Education is the process of gaining knowledge,..."
1,How does education help students?,2,Educational technology improves the learning\n...
2,How does education help students?,3,Student assessment is an important part of\ned...
3,How does education help students?,4,Traditional classroom learning allows students...
4,How does education help students?,5,Inclusive education ensures that all students\...
5,What are the benefits of online learning?,1,Online learning allows students to study throu...
6,What are the benefits of online learning?,2,Educational technology improves the learning\n...
7,What are the benefits of online learning?,3,Distance education allows students to learn\nw...
8,What are the benefits of online learning?,4,Traditional classroom learning allows students...
9,What are the benefits of online learning?,5,"Education is the process of gaining knowledge,..."


Week 11 – Vector Embeddings with ChromaDB

Project Description

This project demonstrates document embedding generation, vector storage, and semantic search using Python and ChromaDB.

Technologies Used

- Python
- NumPy
- Pandas
- Sentence Transformers
- ChromaDB

Embedding Model

"all-MiniLM-L6-v2"

Project Structure

Vector_Embeddings/
│
├── dataset/
│   ├── documents/
│   ├── embeddings.npy
│   ├── queries.csv
│   └── retrieved_results.csv
│
├── chroma_db/
├── notebook/
│   └── Week11_Vector_Embeddings.ipynb
│
├── vector_store.py
├── observations.txt
├── search_comparison.csv
├── README.md
└── requirements.txt

Features

1. Load text documents.
2. Generate vector embeddings.
3. Store embeddings in ChromaDB.
4. Perform semantic search.
5. Perform keyword search.
6. Compare search results.
7. Retrieve top-5 documents for each query.

Execution

Run the following command:

python vector_store.py

Result

Successfully generated embeddings for 10 documents with 384 dimensions and stored them in ChromaDB.

Conclusion

Vector embeddings help identify semantically similar documents and improve information retrieval.